# CS 203 — Week 8 Lab: Hyperparameter Tuning
**Software Tools and Techniques for AI | IIT Gandhinagar**

---

In this lab you will implement Grid Search and Random Search **from scratch**, compare them against scikit-learn's built-in versions, and then use **Optuna** (Bayesian Optimisation) on real datasets. You will measure how efficiently each method finds good hyperparameters under a fixed evaluation budget.

### What you'll build
- `MyGridSearchCV` — a full from-scratch grid search with cross-validation
- `MyRandomSearchCV` — a full from-scratch random search with cross-validation
- Side-by-side comparisons with `sklearn.model_selection.GridSearchCV` and `RandomizedSearchCV`
- Optuna studies with visualisation of how the optimiser learns over trials
- A budget-controlled experiment comparing all three strategies

### Datasets used
| Dataset | Task | Why |
|---------|------|-----|
| Iris | Classification | Small, clean, good for sanity-checks |
| Breast Cancer Wisconsin | Classification | Binary, meaningful, fast to train |
| Wine | Classification | Multi-class, compact |
| California Housing | Regression | Larger, real-world numeric features |



## 0  Setup

In [ ]:
!pip install optuna --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import itertools
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_iris, load_breast_cancer, load_wine, fetch_california_housing
from sklearn.model_selection import (
    KFold, StratifiedKFold, cross_val_score,
    GridSearchCV, RandomizedSearchCV
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from scipy.stats import randint, uniform

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)
print("All imports successful ✓")

In [ ]:
# Load all datasets once — reuse throughout the lab
iris        = load_iris()
cancer      = load_breast_cancer()
wine        = load_wine()
california  = fetch_california_housing()

X_iris,   y_iris   = iris.data,    iris.target
X_cancer, y_cancer = cancer.data,  cancer.target
X_wine,   y_wine   = wine.data,    wine.target
X_cal,    y_cal    = california.data, california.target

print(f"Iris:            {X_iris.shape}")
print(f"Breast Cancer:   {X_cancer.shape}")
print(f"Wine:            {X_wine.shape}")
print(f"California:      {X_cal.shape}")

---
## Part 1 — Grid Search from Scratch

### 1.1  Implement `MyGridSearchCV`

Your class must:
- Accept any scikit-learn compatible estimator, a parameter grid (dict of `{param_name: [values]}`), and `cv` (number of folds).
- Generate **all combinations** of the parameter grid (`itertools.product` will help).
- For each combination, run k-fold cross-validation and record the mean score.
- Expose:
  - `best_params_` — the parameter dict giving the highest CV score
  - `best_score_` — the corresponding score
  - `cv_results_` — a list of dicts, one per combination, with keys `params` and `mean_test_score`

You must use `sklearn.model_selection.KFold` internally (not `cross_val_score` — build the fold loop yourself).

In [ ]:
class MyGridSearchCV:
    """From-scratch grid search with k-fold cross-validation."""

    def __init__(self, estimator, param_grid, cv=5, scoring='accuracy'):
        self.estimator  = estimator
        self.param_grid = param_grid
        self.cv         = cv
        self.scoring    = scoring
        self.best_params_ = None
        self.best_score_  = -np.inf
        self.cv_results_  = []

    def _get_all_combinations(self):
        """Return a list of dicts, each representing one parameter combination."""
        keys   = list(self.param_grid.keys())
        values = list(self.param_grid.values())
        return [dict(zip(keys, combo)) for combo in itertools.product(*values)]

    def _score_single_combination(self, params, X, y):
        """Run k-fold CV for one parameter combination; return mean accuracy."""
        from sklearn.base import clone
        estimator = clone(self.estimator)
        estimator.set_params(**params)
        kf = KFold(n_splits=self.cv, shuffle=True, random_state=SEED)
        fold_scores = []
        for train_idx, val_idx in kf.split(X):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            estimator.fit(X_train, y_train)
            fold_scores.append(estimator.score(X_val, y_val))
        return np.mean(fold_scores)

    def fit(self, X, y):
        """Run the grid search. Populates best_params_, best_score_, cv_results_."""
        for params in self._get_all_combinations():
            score = self._score_single_combination(params, X, y)
            self.cv_results_.append({'params': params, 'mean_test_score': score})
            if score > self.best_score_:
                self.best_score_  = score
                self.best_params_ = params
        return self

    def get_results_df(self):
        """Return cv_results_ as a sorted DataFrame."""
        df = pd.DataFrame(self.cv_results_)
        return df.sort_values('mean_test_score', ascending=False).reset_index(drop=True)


### 1.2  Test your implementation on the Iris dataset

In [ ]:
param_grid_dt = {
    'max_depth':        [1, 2, 3, 5, 7, 10],
    'min_samples_leaf': [1, 2, 4],
}

my_gs = MyGridSearchCV(
    estimator  = DecisionTreeClassifier(random_state=SEED),
    param_grid = param_grid_dt,
    cv         = 5
)
my_gs.fit(X_iris, y_iris)

print(f"Best params : {my_gs.best_params_}")
print(f"Best CV score: {my_gs.best_score_:.4f}")
print()
print(my_gs.get_results_df().head(8))

### 1.3  Compare with sklearn's `GridSearchCV`

Run sklearn's `GridSearchCV` on the **same** dataset with the **same** parameter grid and `cv=5`. Then fill in the comparison table below.

In [ ]:
# Run sklearn GridSearchCV with the same estimator, param_grid_dt, and cv=5
sk_gs = GridSearchCV(
    estimator  = DecisionTreeClassifier(random_state=SEED),
    param_grid = param_grid_dt,
    cv         = 5
)
sk_gs.fit(X_iris, y_iris)

print(f"[sklearn] Best params : {sk_gs.best_params_}")
print(f"[sklearn] Best CV score: {sk_gs.best_score_:.4f}")
print()
print(f"[mine]   Best params : {my_gs.best_params_}")
print(f"[mine]   Best CV score: {my_gs.best_score_:.4f}")


In [ ]:
# Bar chart of mean CV score for every parameter combination, coloured by max_depth
df_results = my_gs.get_results_df()
df_results['max_depth']        = df_results['params'].apply(lambda p: p['max_depth'])
df_results['min_samples_leaf'] = df_results['params'].apply(lambda p: p['min_samples_leaf'])
df_results['label'] = df_results.apply(
    lambda r: f"d={r['max_depth']}, msl={r['min_samples_leaf']}", axis=1
)

depths        = sorted(df_results['max_depth'].unique())
colour_map    = {d: plt.cm.tab10(i / len(depths)) for i, d in enumerate(depths)}
colours       = df_results['max_depth'].map(colour_map)

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(df_results)), df_results['mean_test_score'], color=colours)
ax.set_xticks(range(len(df_results)))
ax.set_xticklabels(df_results['label'], rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Mean CV Accuracy')
ax.set_title('Grid Search Results on Iris — coloured by max_depth')
ax.set_ylim(0.85, 1.0)
from matplotlib.patches import Patch
legend_handles = [Patch(color=colour_map[d], label=f'depth={d}') for d in depths]
ax.legend(handles=legend_handles, title='max_depth', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()


### 1.4  Grid Search on Breast Cancer — two hyperparameters, an SVM

Grid search an SVM (`SVC`) on the Breast Cancer dataset over the following grid:

```python
param_grid_svm = {
    'C':      [0.01, 0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
}
```

Wrap the SVM in a `Pipeline` with a `StandardScaler` **inside** the pipeline (no leakage!). Report the best params and score. Then plot a **heatmap** of mean CV score (rows = `C`, columns = `kernel`).

In [ ]:
param_grid_svm = {
    'svm__C':      [0.01, 0.1, 1, 10, 100],
    'svm__kernel': ['linear', 'rbf'],
}

pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(random_state=SEED))
])

gs_svm = GridSearchCV(pipe_svm, param_grid_svm, cv=5, n_jobs=-1)
gs_svm.fit(X_cancer, y_cancer)

print(f"Best params : {gs_svm.best_params_}")
print(f"Best CV score: {gs_svm.best_score_:.4f}")


In [ ]:
# Heatmap: rows = C values, columns = kernel, colour = mean_test_score
results_df = pd.DataFrame(gs_svm.cv_results_)
results_df['C']      = results_df['params'].apply(lambda p: p['svm__C'])
results_df['kernel'] = results_df['params'].apply(lambda p: p['svm__kernel'])

pivot = results_df.pivot(index='C', columns='kernel', values='mean_test_score')

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(pivot.values, cmap='YlGn', aspect='auto')
plt.colorbar(im, ax=ax, label='Mean CV Accuracy')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)));   ax.set_yticklabels(pivot.index)
ax.set_xlabel('Kernel'); ax.set_ylabel('C')
ax.set_title('SVM Grid Search — Breast Cancer')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f"{pivot.values[i,j]:.3f}", ha='center', va='center', fontsize=9)
plt.tight_layout(); plt.show()


### 1.5  The Curse of Dimensionality — counting evaluations

The grid below is for a Random Forest:

```python
param_grid_rf_large = {
    'n_estimators':    [50, 100, 200, 300, 500],
    'max_depth':       [3, 5, 10, 15, None],
    'min_samples_leaf':[1, 2, 5, 10],
    'max_features':    ['sqrt', 'log2', 0.3, 0.5],
}
```

**Without running any code**, compute:
1. How many unique combinations are there?
2. With `cv=5`, how many total model fits does grid search require?
3. If each fit takes 0.5 seconds, how many minutes would this take?

Then confirm programmatically:

In [ ]:
param_grid_rf_large = {
    'n_estimators':    [50, 100, 200, 300, 500],
    'max_depth':       [3, 5, 10, 15, None],
    'min_samples_leaf':[1, 2, 5, 10],
    'max_features':    ['sqrt', 'log2', 0.3, 0.5],
}

n_combos = 1
for v in param_grid_rf_large.values():
    n_combos *= len(v)

cv_folds      = 5
total_fits    = n_combos * cv_folds
total_minutes = total_fits * 0.5 / 60

print(f"Unique combinations : {n_combos}")
print(f"Total model fits    : {total_fits}  (= {n_combos} combos x cv={cv_folds})")
print(f"Estimated time      : {total_minutes:.1f} minutes  (at 0.5 s per fit)")


---
## Part 2 — Random Search from Scratch

### 2.1  Implement `MyRandomSearchCV`

Your class must:
- Accept the same interface as `MyGridSearchCV`, **plus** `n_iter` (number of random samples) and `random_state`.
- The parameter space can contain either a **list** of values (sample uniformly from that list) or a **scipy distribution** with a `.rvs()` method.
- For each of `n_iter` iterations, sample one value per parameter, run k-fold CV, and record the result.
- Expose the same attributes: `best_params_`, `best_score_`, `cv_results_`.

In [ ]:
class MyRandomSearchCV:
    """From-scratch random search with k-fold cross-validation."""

    def __init__(self, estimator, param_distributions, n_iter=20,
                 cv=5, scoring='accuracy', random_state=None):
        self.estimator           = estimator
        self.param_distributions = param_distributions
        self.n_iter              = n_iter
        self.cv                  = cv
        self.scoring             = scoring
        self.random_state        = random_state
        self.best_params_ = None
        self.best_score_  = -np.inf
        self.cv_results_  = []

    def _sample_params(self, rng):
        """Sample one value per hyperparameter."""
        params = {}
        for key, dist in self.param_distributions.items():
            if isinstance(dist, list):
                params[key] = rng.choice(dist)
            else:
                # scipy distribution — .rvs(random_state=rng)
                val = dist.rvs(random_state=rng)
                params[key] = int(val) if hasattr(val, '__int__') else val
        return params

    def _cross_val_score(self, params, X, y):
        """K-fold CV for one sampled combination; return mean accuracy."""
        from sklearn.base import clone
        estimator = clone(self.estimator)
        estimator.set_params(**params)
        kf = KFold(n_splits=self.cv, shuffle=True, random_state=SEED)
        fold_scores = []
        for train_idx, val_idx in kf.split(X):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            estimator.fit(X_train, y_train)
            fold_scores.append(estimator.score(X_val, y_val))
        return np.mean(fold_scores)

    def fit(self, X, y):
        """Run n_iter random trials. Populates best_params_, best_score_, cv_results_."""
        rng = np.random.RandomState(self.random_state)
        for _ in range(self.n_iter):
            params = self._sample_params(rng)
            score  = self._cross_val_score(params, X, y)
            self.cv_results_.append({'params': params, 'mean_test_score': score})
            if score > self.best_score_:
                self.best_score_  = score
                self.best_params_ = params
        return self

    def get_results_df(self):
        df = pd.DataFrame(self.cv_results_)
        return df.sort_values('mean_test_score', ascending=False).reset_index(drop=True)


### 2.2  Test on Iris

In [ ]:
param_dist_dt = {
    'max_depth':        [1, 2, 3, 5, 7, 10, 15, 20],
    'min_samples_leaf': randint(1, 10),     # scipy distribution
    'min_samples_split': randint(2, 20),
}

my_rs = MyRandomSearchCV(
    estimator            = DecisionTreeClassifier(random_state=SEED),
    param_distributions  = param_dist_dt,
    n_iter               = 30,
    cv                   = 5,
    random_state         = SEED
)
my_rs.fit(X_iris, y_iris)

print(f"Best params : {my_rs.best_params_}")
print(f"Best CV score: {my_rs.best_score_:.4f}")
print()
print(my_rs.get_results_df().head(8))

### 2.3  Compare with sklearn's `RandomizedSearchCV`

Run sklearn's version with the **same** parameter distributions, `n_iter=30`, `cv=5`, and `random_state=SEED`. Print both best scores and params side by side.

In [ ]:
# sklearn RandomizedSearchCV — same distributions, same n_iter, same seed
sk_rs = RandomizedSearchCV(
    estimator           = DecisionTreeClassifier(random_state=SEED),
    param_distributions = param_dist_dt,
    n_iter              = 30,
    cv                  = 5,
    random_state        = SEED
)
sk_rs.fit(X_iris, y_iris)

print(f"[sklearn] Best params : {sk_rs.best_params_}")
print(f"[sklearn] Best CV score: {sk_rs.best_score_:.4f}")
print()
print(f"[mine]   Best params : {my_rs.best_params_}")
print(f"[mine]   Best CV score: {my_rs.best_score_:.4f}")


### 2.4  Convergence plot — best score vs number of trials

For your `MyRandomSearchCV`, plot how the **best score seen so far** evolves as you go from trial 1 to trial 30. Do the same for your `MyGridSearchCV` results (pretend they arrive in a fixed order).

Hint: compute `np.maximum.accumulate` on the list of scores in the order they were evaluated.

In [ ]:
# Extract scores in evaluation order
rs_scores = [r['mean_test_score'] for r in my_rs.cv_results_]
gs_scores = [r['mean_test_score'] for r in my_gs.cv_results_]

rs_cummax = np.maximum.accumulate(rs_scores)
gs_cummax = np.maximum.accumulate(gs_scores)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, len(rs_cummax)+1), rs_cummax, marker='o', markersize=3,
        label='Random Search (cumulative best)', color='steelblue')
ax.plot(range(1, len(gs_cummax)+1), gs_cummax, marker='s', markersize=3,
        label='Grid Search (cumulative best)',   color='darkorange')
ax.scatter(range(1, len(rs_scores)+1), rs_scores, alpha=0.3, s=15, color='steelblue')
ax.scatter(range(1, len(gs_scores)+1), gs_scores, alpha=0.3, s=15, color='darkorange')
ax.set_xlabel('Evaluation #'); ax.set_ylabel('CV Accuracy')
ax.set_title('Convergence: Best Score Seen So Far (Iris, Decision Tree)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


---
## Part 3 — Grid vs Random: The Bergstra & Bengio Experiment

The lecture showed that random search covers the important dimension more densely than grid search when one hyperparameter matters much more than the other. Here you will **reproduce that intuition empirically**.

### 3.1  Fixed budget comparison on Wine dataset

Budget = **25 evaluations** with `cv=5`. Use a `RandomForestClassifier`.

- Grid search: design a grid with exactly 25 combinations (e.g., 5 × 5 or 5 × 5 × 1).
- Random search: 25 random samples from a **wide** continuous distribution.

Report: best score, best params, and wall-clock time for each.

In [ ]:
BUDGET = 25

# Grid search — exactly 25 combinations (5 x 5)
param_grid_wine = {
    'n_estimators': [50, 100, 150, 200, 300],
    'max_depth':    [3, 5, 8, 12, None],
}

n_combos = 1
for v in param_grid_wine.values():
    n_combos *= len(v)
assert n_combos == BUDGET, f"Expected {BUDGET} combos, got {n_combos}"

t0 = time.time()
gs_wine = GridSearchCV(
    RandomForestClassifier(random_state=SEED),
    param_grid_wine, cv=5, n_jobs=-1
)
gs_wine.fit(X_wine, y_wine)
t_grid = time.time() - t0

print(f"[Grid]   best={gs_wine.best_score_:.4f}  params={gs_wine.best_params_}  time={t_grid:.2f}s")


In [ ]:
# Random search — 25 iterations, wide continuous distributions
param_dist_wine = {
    'n_estimators':      randint(10, 500),       # wide integer range
    'max_depth':         randint(2, 30),
    'min_samples_leaf':  randint(1, 20),
    'max_features':      uniform(0.1, 0.9),      # continuous [0.1, 1.0]
}

t0 = time.time()
rs_wine = RandomizedSearchCV(
    RandomForestClassifier(random_state=SEED),
    param_dist_wine, n_iter=BUDGET, cv=5, n_jobs=-1, random_state=SEED
)
rs_wine.fit(X_wine, y_wine)
t_rand = time.time() - t0

print(f"[Random] best={rs_wine.best_score_:.4f}  params={rs_wine.best_params_}  time={t_rand:.2f}s")


### 3.2  Coverage plot

Pick two hyperparameters (e.g., `n_estimators` and `max_depth`). Scatter-plot all 25 evaluated points for grid search and random search on the same axes (different colours). What do you observe?

In [ ]:
# Coverage scatter plot — n_estimators vs max_depth
gs_ne = [p['n_estimators'] for p in gs_wine.cv_results_['params']]
gs_md = [p['max_depth']    for p in gs_wine.cv_results_['params']]
gs_md_plot = [d if d is not None else 30 for d in gs_md]   # None -> 30

rs_ne = [p['n_estimators'] for p in rs_wine.cv_results_['params']]
rs_md = [p['max_depth']    for p in rs_wine.cv_results_['params']]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(gs_ne, gs_md_plot, c='darkorange', s=90, marker='s',
           label='Grid Search', edgecolors='k', linewidths=0.4)
ax.scatter(rs_ne, rs_md,      c='steelblue',  s=90, marker='o',
           label='Random Search', edgecolors='k', linewidths=0.4, alpha=0.75)
ax.set_xlabel('n_estimators'); ax.set_ylabel('max_depth  (30 = None)')
ax.set_title(f'Coverage of Hyperparameter Space ({BUDGET} evaluations each)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("Observation: Grid search uses a regular lattice; random search covers")
print("the continuous range more evenly — confirming Bergstra & Bengio (2012).")


### 3.3  Repeated trials — mean and standard deviation of best score

Random search has variance across different random seeds. Repeat it 10 times (with seeds 0–9) and report the mean ± std of `best_score_`. Do the same for your `MyRandomSearchCV`.

In [ ]:
sklearn_scores = []
my_scores      = []

for seed in range(10):
    # sklearn RandomizedSearchCV with this seed
    rs_tmp = RandomizedSearchCV(
        RandomForestClassifier(random_state=SEED),
        param_dist_wine, n_iter=BUDGET, cv=5, n_jobs=-1, random_state=seed
    )
    rs_tmp.fit(X_wine, y_wine)
    sklearn_scores.append(rs_tmp.best_score_)

    # MyRandomSearchCV with this seed
    my_tmp = MyRandomSearchCV(
        RandomForestClassifier(random_state=SEED),
        param_dist_wine, n_iter=BUDGET, cv=5, random_state=seed
    )
    my_tmp.fit(X_wine, y_wine)
    my_scores.append(my_tmp.best_score_)

print(f"[sklearn] mean={np.mean(sklearn_scores):.4f}  std={np.std(sklearn_scores):.4f}")
print(f"[mine]   mean={np.mean(my_scores):.4f}  std={np.std(my_scores):.4f}")


---
## Part 4 — Bayesian Optimisation with Optuna

### 4.1  Your first Optuna study

Write an Optuna objective function for a `RandomForestClassifier` on the Breast Cancer dataset. Tune:
- `n_estimators`: integer between 50 and 500
- `max_depth`: integer between 2 and 30
- `min_samples_leaf`: integer between 1 and 20
- `max_features`: float between 0.1 and 1.0

Run for 60 trials. Print the best value and best params.

In [ ]:
def objective_cancer(trial):
    """Optuna objective: return the 5-fold CV accuracy on Breast Cancer."""
    n_estimators     = trial.suggest_int('n_estimators',    50, 500)
    max_depth        = trial.suggest_int('max_depth',        2,  30)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1,  20)
    max_features     = trial.suggest_float('max_features',  0.1, 1.0)

    model = RandomForestClassifier(
        n_estimators     = n_estimators,
        max_depth        = max_depth,
        min_samples_leaf = min_samples_leaf,
        max_features     = max_features,
        random_state     = SEED
    )
    cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    scores = cross_val_score(model, X_cancer, y_cancer, cv=cv, scoring='accuracy')
    return scores.mean()


study_cancer = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_cancer.optimize(objective_cancer, n_trials=60)

print(f"Best CV accuracy : {study_cancer.best_value:.4f}")
print(f"Best params      : {study_cancer.best_params}")


### 4.2  Visualise how Optuna learns over trials

Plot the **objective value per trial** and the **cumulative best** on the same figure. The cumulative best should be monotonically non-decreasing.

In [ ]:
# Per-trial and cumulative-best plot
trial_numbers   = [t.number for t in study_cancer.trials]
trial_values    = [t.value  for t in study_cancer.trials]
cumulative_best = np.maximum.accumulate(trial_values)

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(trial_numbers, trial_values, alpha=0.5, s=20,
           color='steelblue', label='Per-trial objective')
ax.plot(trial_numbers, cumulative_best, color='crimson', linewidth=2,
        label='Cumulative best')
ax.set_xlabel('Trial number'); ax.set_ylabel('CV Accuracy')
ax.set_title('Optuna Optimisation History (Breast Cancer — RF)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


### 4.3  Parameter importance

Optuna can compute how much each hyperparameter contributed to the objective variance. Use `optuna.importance.get_param_importances` and plot a horizontal bar chart.

In [ ]:
importances   = optuna.importance.get_param_importances(study_cancer)
params_list   = list(importances.keys())
scores_list   = list(importances.values())

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(params_list, scores_list, color='steelblue', edgecolor='k', linewidth=0.4)
ax.bar_label(bars, fmt='{:.3f}', padding=3)
ax.set_xlabel('Importance')
ax.set_title('Hyperparameter Importance — Optuna (Breast Cancer RF)')
ax.invert_yaxis()
plt.tight_layout(); plt.show()


### 4.4  Optuna on a regression task — California Housing

Switch to regression. Tune a `GradientBoostingRegressor` on the California Housing dataset. Maximise the negative RMSE (or equivalently minimise RMSE — set `direction='minimize'` and return the RMSE).

Hyperparameters to tune:
- `n_estimators`: 50 – 400
- `learning_rate`: log-uniform between 1e-3 and 1.0 (use `trial.suggest_float(..., log=True)`)
- `max_depth`: 2 – 10
- `subsample`: 0.5 – 1.0

Run 50 trials.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score as cvs

def objective_cal(trial):
    """Return RMSE (to be minimised)."""
    n_estimators  = trial.suggest_int('n_estimators',   50, 400)
    learning_rate = trial.suggest_float('learning_rate', 1e-3, 1.0, log=True)
    max_depth     = trial.suggest_int('max_depth',        2,  10)
    subsample     = trial.suggest_float('subsample',      0.5, 1.0)

    model = GradientBoostingRegressor(
        n_estimators  = n_estimators,
        learning_rate = learning_rate,
        max_depth     = max_depth,
        subsample     = subsample,
        random_state  = SEED
    )
    kf       = KFold(n_splits=5, shuffle=True, random_state=SEED)
    neg_rmse = cvs(model, X_cal, y_cal,
                   cv=kf, scoring='neg_root_mean_squared_error')
    return -neg_rmse.mean()   # positive RMSE to minimise


study_cal = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_cal.optimize(objective_cal, n_trials=50)

print(f"Best RMSE   : {study_cal.best_value:.4f}")
print(f"Best params : {study_cal.best_params}")


---
## Part 5 — Head-to-Head: Grid vs Random vs BayesOpt

### 5.1  Benchmark under a shared evaluation budget

Use the **Wine** dataset with a `RandomForestClassifier`. Fix the budget to **N = 40 evaluations** (each evaluation = one (params, cv) pair → 5 model fits). Compare:

| Method | How to fix budget |
|--------|------------------|
| Grid search | Design a grid with ≤ 40 combinations |
| Random search | `n_iter=40` |
| Optuna (BayesOpt) | `n_trials=40` |

Record: best CV score and wall-clock time for each. Report in a summary table.

In [ ]:
BUDGET = 40
results_summary = {}

# --- 1. Grid Search (40 combos: 8 x 5) ---
param_grid_40 = {
    'n_estimators': [50, 100, 150, 200, 250, 300, 400, 500],
    'max_depth':    [3, 5, 10, 15, None],
}
t0 = time.time()
gs_40 = GridSearchCV(RandomForestClassifier(random_state=SEED),
                     param_grid_40, cv=5, n_jobs=-1)
gs_40.fit(X_wine, y_wine)
t_gs = time.time() - t0
results_summary['GridSearch'] = {'best_score': gs_40.best_score_, 'time_s': t_gs}

# --- 2. Random Search (40 iterations) ---
param_dist_40 = {
    'n_estimators':     randint(10, 600),
    'max_depth':        randint(2, 30),
    'min_samples_leaf': randint(1, 20),
    'max_features':     uniform(0.1, 0.9),
}
t0 = time.time()
rs_40 = RandomizedSearchCV(RandomForestClassifier(random_state=SEED),
                            param_dist_40, n_iter=BUDGET, cv=5,
                            n_jobs=-1, random_state=SEED)
rs_40.fit(X_wine, y_wine)
t_rs = time.time() - t0
results_summary['RandomSearch'] = {'best_score': rs_40.best_score_, 'time_s': t_rs}

# --- 3. Optuna / BayesOpt (40 trials) ---
def obj_wine_bench(trial):
    n_est = trial.suggest_int('n_estimators',     10, 600)
    depth = trial.suggest_int('max_depth',         2,  30)
    msl   = trial.suggest_int('min_samples_leaf',  1,  20)
    mf    = trial.suggest_float('max_features',   0.1, 1.0)
    model = RandomForestClassifier(n_estimators=n_est, max_depth=depth,
                                   min_samples_leaf=msl, max_features=mf,
                                   random_state=SEED)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    return cross_val_score(model, X_wine, y_wine, cv=cv).mean()

t0 = time.time()
study_wine = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
study_wine.optimize(obj_wine_bench, n_trials=BUDGET)
t_opt = time.time() - t0
results_summary['BayesOpt'] = {'best_score': study_wine.best_value, 'time_s': t_opt}

# Summary table
print(f"{'Method':<15} {'Best CV Score':>13} {'Time (s)':>10}")
print('-' * 42)
for method, vals in results_summary.items():
    print(f"{method:<15} {vals['best_score']:>13.4f} {vals['time_s']:>10.2f}")


### 5.2  Learning curves — best score vs number of evaluations

For the same experiment above, plot **cumulative best score** as a function of trial number (1 to 40) for all three methods on the same axes. This is the canonical way to compare search strategies.

Hint: for Optuna, use `[t.value for t in study.trials]` to get the per-trial scores.

In [ ]:
# Extract per-evaluation scores (in order) for all three methods
gs_scores_40  = list(gs_40.cv_results_['mean_test_score'])
rs_scores_40  = list(rs_40.cv_results_['mean_test_score'])
opt_scores_40 = [t.value for t in study_wine.trials]

# Cumulative best
gs_cum  = np.maximum.accumulate(gs_scores_40)
rs_cum  = np.maximum.accumulate(rs_scores_40)
opt_cum = np.maximum.accumulate(opt_scores_40)

fig, ax = plt.subplots(figsize=(10, 5))
trials = range(1, BUDGET + 1)
ax.plot(trials, gs_cum,  label='Grid Search',    color='darkorange', linewidth=2)
ax.plot(trials, rs_cum,  label='Random Search',  color='steelblue',  linewidth=2)
ax.plot(trials, opt_cum, label='BayesOpt (Optuna)', color='green',   linewidth=2)
ax.scatter(trials, gs_scores_40,  alpha=0.3, s=15, color='darkorange')
ax.scatter(trials, rs_scores_40,  alpha=0.3, s=15, color='steelblue')
ax.scatter(trials, opt_scores_40, alpha=0.3, s=15, color='green')
ax.set_xlabel('Number of evaluations'); ax.set_ylabel('Best CV score so far')
ax.set_title(f'Learning Curves — Wine Dataset (budget = {BUDGET})')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


### 5.3  Multi-dataset benchmark

Repeat the 3-way comparison (budget = 30) for **all four datasets** (Iris, Breast Cancer, Wine, and for California Housing use negative RMSE as the score). Produce a summary DataFrame with rows = datasets and columns = methods.

In [ ]:
BUDGET = 30
datasets = {
    'Iris':         (X_iris,   y_iris,   'classification'),
    'BreastCancer': (X_cancer, y_cancer, 'classification'),
    'Wine':         (X_wine,   y_wine,   'classification'),
    'California':   (X_cal,    y_cal,    'regression'),
}

bench_results = {}

for ds_name, (X, y, task) in datasets.items():
    print(f"Processing {ds_name}...", end=' ', flush=True)
    scoring = 'accuracy' if task == 'classification' else 'neg_root_mean_squared_error'
    direction = 'maximize' if task == 'classification' else 'minimize'
    sign = 1 if task == 'classification' else -1   # flip sign for regression

    # Grid Search
    if task == 'classification':
        pg = {'n_estimators': [50,100,200,300,500], 'max_depth': [3,5,None]}
        model_cls = RandomForestClassifier
    else:
        from sklearn.ensemble import GradientBoostingRegressor
        pg = {'n_estimators': [50,100,200], 'max_depth': [2,3,5], 'learning_rate':[0.05,0.1]}
        model_cls = GradientBoostingRegressor

    est = model_cls(random_state=SEED)
    gs_b = GridSearchCV(est, pg, cv=5, n_jobs=-1, scoring=scoring)
    gs_b.fit(X, y)
    gs_best = gs_b.best_score_ * sign

    # Random Search
    if task == 'classification':
        pd_b = {'n_estimators': randint(10,500), 'max_depth': randint(2,30),
                'min_samples_leaf': randint(1,20), 'max_features': uniform(0.1,0.9)}
        est2 = RandomForestClassifier(random_state=SEED)
    else:
        pd_b = {'n_estimators': randint(50,400), 'max_depth': randint(2,10),
                'learning_rate': uniform(0.01,0.3), 'subsample': uniform(0.5,0.5)}
        est2 = GradientBoostingRegressor(random_state=SEED)

    rs_b = RandomizedSearchCV(est2, pd_b, n_iter=BUDGET, cv=5,
                               n_jobs=-1, scoring=scoring, random_state=SEED)
    rs_b.fit(X, y)
    rs_best = rs_b.best_score_ * sign

    # Optuna
    def make_obj(X_, y_, task_):
        def obj(trial):
            if task_ == 'classification':
                m = RandomForestClassifier(
                    n_estimators     = trial.suggest_int('n_est',  10, 500),
                    max_depth        = trial.suggest_int('depth',   2,  30),
                    min_samples_leaf = trial.suggest_int('msl',     1,  20),
                    max_features     = trial.suggest_float('mf',  0.1, 1.0),
                    random_state=SEED)
                sc = 'accuracy'
            else:
                from sklearn.ensemble import GradientBoostingRegressor as GBR
                m = GBR(
                    n_estimators  = trial.suggest_int('n_est',   50, 400),
                    learning_rate = trial.suggest_float('lr', 1e-3, 0.5, log=True),
                    max_depth     = trial.suggest_int('depth',    2,  10),
                    subsample     = trial.suggest_float('sub',  0.5, 1.0),
                    random_state=SEED)
                sc = 'neg_root_mean_squared_error'
            return cross_val_score(m, X_, y_, cv=5, scoring=sc).mean()
        return obj

    study_b = optuna.create_study(direction=direction,
                                  sampler=optuna.samplers.TPESampler(seed=SEED))
    study_b.optimize(make_obj(X, y, task), n_trials=BUDGET)
    opt_best = study_b.best_value * sign

    bench_results[ds_name] = {'Grid': gs_best, 'Random': rs_best, 'BayesOpt': opt_best}
    print("done")

df_bench = pd.DataFrame(bench_results).T
df_bench.index.name = 'Dataset'
print()
print(df_bench.round(4))


---
## Part 6 — Advanced Optuna: Pruning and Categorical Hyperparameters

### 6.1  Categorical hyperparameters — choosing the model family

Use `trial.suggest_categorical` to let Optuna choose **both** the model family **and** its hyperparameters in a single study. Include at least: `DecisionTree`, `RandomForest`, `GradientBoosting`, `KNN`. Run on Breast Cancer for 60 trials.

In [ ]:
def objective_automl(trial):
    """Let Optuna choose the model family and tune it."""
    classifier = trial.suggest_categorical('classifier', ['DT', 'RF', 'GB', 'KNN'])

    if classifier == 'DT':
        model = DecisionTreeClassifier(
            max_depth        = trial.suggest_int('dt_max_depth',        1, 30),
            min_samples_leaf = trial.suggest_int('dt_min_samples_leaf', 1, 20),
            random_state     = SEED
        )
    elif classifier == 'RF':
        model = RandomForestClassifier(
            n_estimators     = trial.suggest_int('rf_n_estimators',     50, 500),
            max_depth        = trial.suggest_int('rf_max_depth',          2,  30),
            min_samples_leaf = trial.suggest_int('rf_min_samples_leaf',   1,  20),
            random_state     = SEED
        )
    elif classifier == 'GB':
        model = GradientBoostingClassifier(
            n_estimators  = trial.suggest_int('gb_n_estimators',   50, 300),
            learning_rate = trial.suggest_float('gb_learning_rate', 1e-3, 1.0, log=True),
            max_depth     = trial.suggest_int('gb_max_depth',        2,  10),
            random_state  = SEED
        )
    else:  # KNN
        model = KNeighborsClassifier(
            n_neighbors = trial.suggest_int('knn_n_neighbors', 1, 30),
            weights     = trial.suggest_categorical('knn_weights', ['uniform', 'distance'])
        )

    cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    scores = cross_val_score(model, X_cancer, y_cancer, cv=cv, scoring='accuracy')
    return scores.mean()


study_automl = optuna.create_study(direction='maximize',
                                   sampler=optuna.samplers.TPESampler(seed=SEED))
study_automl.optimize(objective_automl, n_trials=60)

print(f"Best CV accuracy : {study_automl.best_value:.4f}")
print(f"Best params      : {study_automl.best_params}")


In [ ]:
# Count how many trials each classifier was selected
from collections import Counter
clf_counts = Counter(
    t.params['classifier'] for t in study_automl.trials
    if t.state.name == 'COMPLETE'
)

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(clf_counts.keys(), clf_counts.values(), color='steelblue', edgecolor='k')
ax.set_xlabel('Classifier'); ax.set_ylabel('# Trials selected')
ax.set_title('AutoML — Classifier Family Selection (60 trials)')
plt.tight_layout(); plt.show()

print(dict(clf_counts))


### 6.2  Reproducing a study — `random_state` and Optuna seeds

Re-run your `objective_cancer` study with `n_trials=60` **twice**, both times using the same `TPESampler(seed=SEED)`. Confirm that both runs produce **exactly the same** sequence of trial values. What does this tell you about reproducibility?

In [ ]:
# Run the same study twice — both with TPESampler(seed=SEED)
study_rep1 = optuna.create_study(direction='maximize',
                                  sampler=optuna.samplers.TPESampler(seed=SEED))
study_rep1.optimize(objective_cancer, n_trials=60)

study_rep2 = optuna.create_study(direction='maximize',
                                  sampler=optuna.samplers.TPESampler(seed=SEED))
study_rep2.optimize(objective_cancer, n_trials=60)

vals1 = [t.value for t in study_rep1.trials]
vals2 = [t.value for t in study_rep2.trials]

identical = vals1 == vals2
print(f"Both runs produced identical trial sequences: {identical}")
print(f"Run 1 best: {max(vals1):.4f}")
print(f"Run 2 best: {max(vals2):.4f}")
print()
print("Conclusion: fixing TPESampler(seed) makes Optuna fully reproducible.")


### 6.3  Multi-seed variance report

Run Optuna on the Wine dataset (30 trials each) using 10 different seeds (0–9). Report the mean ± std of the best CV score across seeds. Do the same for Random Search with the same budget.

In [ ]:
optuna_scores = []
random_scores = []

param_dist_wine_6 = {
    'n_estimators':     randint(10, 500),
    'max_depth':        randint(2, 30),
    'min_samples_leaf': randint(1, 20),
    'max_features':     uniform(0.1, 0.9),
}

for seed in range(10):
    # Optuna run
    def obj_wine_s(trial):
        model = RandomForestClassifier(
            n_estimators     = trial.suggest_int('n_est',  10, 500),
            max_depth        = trial.suggest_int('depth',   2,  30),
            min_samples_leaf = trial.suggest_int('msl',     1,  20),
            max_features     = trial.suggest_float('mf',  0.1, 1.0),
            random_state=SEED)
        return cross_val_score(model, X_wine, y_wine, cv=5).mean()

    s = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=seed))
    s.optimize(obj_wine_s, n_trials=30)
    optuna_scores.append(s.best_value)

    # RandomizedSearchCV run
    rs_s = RandomizedSearchCV(
        RandomForestClassifier(random_state=SEED),
        param_dist_wine_6, n_iter=30, cv=5, n_jobs=-1, random_state=seed
    )
    rs_s.fit(X_wine, y_wine)
    random_scores.append(rs_s.best_score_)

print(f"[Optuna] mean={np.mean(optuna_scores):.4f}  std={np.std(optuna_scores):.4f}")
print(f"[Random] mean={np.mean(random_scores):.4f}  std={np.std(random_scores):.4f}")


---
## Part 7 — Pipelines, Preprocessing, and Data Leakage

### 7.1  The wrong way vs the right way

Demonstrate data leakage: scale the entire Breast Cancer dataset **before** running cross-validation, and compare the CV score to scaling **inside** a Pipeline. The scores should differ — explain why.

In [ ]:
from sklearn.svm import SVC

# --- WRONG: scale before CV (leaky) ---
scaler = StandardScaler()
X_cancer_scaled = scaler.fit_transform(X_cancer)  # sees ALL data including future val folds!

leaky_score = cross_val_score(
    SVC(C=1, kernel='rbf', random_state=SEED),
    X_cancer_scaled, y_cancer, cv=5
).mean()

# --- RIGHT: scale inside Pipeline ---
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(C=1, kernel='rbf', random_state=SEED))
])
correct_score = cross_val_score(pipe, X_cancer, y_cancer, cv=5).mean()

print(f"Leaky  (wrong) CV accuracy: {leaky_score:.4f}")
print(f"Correct (Pipeline) accuracy: {correct_score:.4f}")
print(f"Difference: {leaky_score - correct_score:+.4f}")

### 7.2  Tuning a Pipeline with `GridSearchCV`

Build a `Pipeline(StandardScaler → SVC)` and tune it with `GridSearchCV` on Breast Cancer. Use the `svm__` prefix for the SVM parameters.

```python
param_grid_pipe = {
    'svm__C':      [0.01, 0.1, 1, 10, 100],
    'svm__kernel': ['linear', 'rbf'],
    'svm__gamma':  ['scale', 'auto'],
}
```

In [ ]:
param_grid_pipe = {
    'svm__C':      [0.01, 0.1, 1, 10, 100],
    'svm__kernel': ['linear', 'rbf'],
    'svm__gamma':  ['scale', 'auto'],
}

pipe2 = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(random_state=SEED))
])

gs_pipe = GridSearchCV(pipe2, param_grid_pipe, cv=5, n_jobs=-1)
gs_pipe.fit(X_cancer, y_cancer)

print(f"Best params : {gs_pipe.best_params_}")
print(f"Best CV score: {gs_pipe.best_score_:.4f}")


---
## Part 8 — Capstone: Full Tuning Workflow

Implement the complete tuning workflow from the lecture on the **Wine** dataset:

1. **Dummy baseline** — what does a random classifier score?
2. **Simple model** — Logistic Regression with default params.
3. **Tuned strong model** — Random Forest, tuned with Optuna (50 trials).
4. **DIY AutoML** — loop over at least 4 model families, each with its own grid, pick the winner.

Print a clearly formatted leaderboard at the end.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import ExtraTreesClassifier

leaderboard = {}

# 1. Dummy baseline
dummy = DummyClassifier(strategy='most_frequent', random_state=SEED)
leaderboard['DummyClassifier'] = cross_val_score(
    dummy, X_wine, y_wine, cv=5).mean()

# 2. Logistic Regression (default)
lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(max_iter=1000, random_state=SEED))
])
leaderboard['LogisticRegression (default)'] = cross_val_score(
    lr, X_wine, y_wine, cv=5).mean()

# 3. Random Forest tuned with Optuna (50 trials)
def obj_wine_cap(trial):
    model = RandomForestClassifier(
        n_estimators     = trial.suggest_int('n_estimators',    50, 600),
        max_depth        = trial.suggest_int('max_depth',         2,  30),
        min_samples_leaf = trial.suggest_int('min_samples_leaf',  1,  20),
        max_features     = trial.suggest_float('max_features',  0.1, 1.0),
        random_state     = SEED
    )
    return cross_val_score(model, X_wine, y_wine, cv=5).mean()

study_cap = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
study_cap.optimize(obj_wine_cap, n_trials=50)
leaderboard['RandomForest (Optuna 50 trials)'] = study_cap.best_value

# 4. DIY AutoML — 4 model families
model_configs = {
    'DecisionTree': (
        DecisionTreeClassifier(random_state=SEED),
        {'max_depth': [3, 5, 10, None], 'min_samples_leaf': [1, 2, 5]}
    ),
    'SVM': (
        Pipeline([('sc', StandardScaler()), ('svm', SVC(random_state=SEED))]),
        {'svm__C': [0.1, 1, 10, 100], 'svm__kernel': ['linear', 'rbf']}
    ),
    'KNN': (
        Pipeline([('sc', StandardScaler()), ('knn', KNeighborsClassifier())]),
        {'knn__n_neighbors': [3, 5, 7, 11, 15]}
    ),
    'ExtraTrees': (
        ExtraTreesClassifier(random_state=SEED),
        {'n_estimators': [50, 100, 200], 'max_depth': [5, 10, None]}
    ),
}

for name, (model, params) in model_configs.items():
    gs = GridSearchCV(model, params, cv=5, n_jobs=-1)
    gs.fit(X_wine, y_wine)
    leaderboard[f'AutoML-{name}'] = gs.best_score_

# Leaderboard
print(f"\n{'Method':<40} {'CV Score':>10}")
print('─' * 52)
for method, score in sorted(leaderboard.items(), key=lambda x: x[1], reverse=True):
    print(f"{method:<40} {score:>10.4f}")


### 8.1  Reflection questions

Answer the following in the markdown cell below (no code needed):

1. Your `MyGridSearchCV` and sklearn's `GridSearchCV` should give the same best score. If they differ slightly, what could cause the discrepancy?
2. In the convergence plot (Part 5.2), which method found a good score with the fewest evaluations? What does this tell you?
3. A colleague suggests running Optuna with 500 trials on every project. Give two reasons why this might be wasteful.
4. Why does putting `StandardScaler` inside a `Pipeline` prevent data leakage during cross-validation?
5. Looking at the parameter importance chart (Part 4.3), which hyperparameter mattered most? Does this match your intuition?

*(Write your answers here)*

1. **Possible discrepancy between `MyGridSearchCV` and sklearn's `GridSearchCV`:**  
   Both should match in score if the same `random_state` is used for `KFold`. Small differences can arise from: (a) different default shuffling—sklearn's `GridSearchCV` uses `cv=5` with no shuffle by default (it uses the data order), while my implementation calls `KFold(shuffle=True, random_state=SEED)`; (b) floating-point ordering when multiple param combos tie for best score.

2. **Which method found a good score with fewest evaluations?**  
   BayesOpt (Optuna) typically reaches a high score fastest—it uses past results to propose the next hyperparameter configuration, so it converges in fewer evaluations than random search, which converges faster than grid search.

3. **Two reasons 500 Optuna trials might be wasteful:**  
   (a) Diminishing returns—the TPE surrogate usually converges well before 500 trials on typical ML problems; extra trials rarely improve the best score.  
   (b) Computational cost—each trial involves a full cross-validation, so 500 trials × 5 folds = 2500 model fits, which can take hours on large datasets.

4. **Why `StandardScaler` inside a `Pipeline` prevents data leakage:**  
   When `cross_val_score` uses a `Pipeline`, it re-fits the entire pipeline (including the scaler) on each training fold. The scaler only sees training-fold data to compute mean/std, so the validation fold is never seen during fitting. Scaling before CV leaks validation statistics into the scaler, giving an optimistically biased estimate.

5. **Most important hyperparameter from the importance chart:**  
   Typically `n_estimators` or `max_features` dominates. `max_features` is often most influential for Random Forests because it controls the diversity of trees—low values increase variance; high values reduce diversity. This matches the literature on RF sensitivity.


---
## Bonus — Optuna Visualisation Dashboard

Optuna ships built-in interactive plots. Run the cells below on any completed study to explore the results visually (works best in a Jupyter/Colab environment with `plotly` installed).

In [ ]:
!pip install plotly --quiet

In [ ]:
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
    plot_contour,
)

# Change `study_cancer` to any study you've created above
plot_optimization_history(study_cancer).show()
plot_param_importances(study_cancer).show()
plot_parallel_coordinate(study_cancer).show()

In [ ]:
# Contour plot for two parameters of your choice
plot_contour(study_cancer, params=['n_estimators', 'max_depth']).show()